# Grupo 9 — Lending Club: Exploración inicial y baseline

CS3061 Machine Learning · Entrega previa (Avance 1)
Integrantes:
- Pedro Nicolás Solís Córdova - 202510793
- Guillermo Valentino Ceceu Gamboa Sánchez - 202510741
- Miranda Lucia Salazar Cabeza 202210310
- Rubi Marie ludeña Molero 202110200

Este notebook consolida el trabajo de los 4 integrantes: carga y calidad de datos (Nicolás), exploración visual y outliers (Kris), leakage y sesgos (Rubí), y partición/baseline (Valentino).

## 0. Configuración: descarga del dataset desde Kaggle

Suba su archivo `kaggle.json` cuando la primera celda lo solicite.

In [ ]:
from google.colab import files
files.upload()


In [ ]:
import os
os.makedirs("/root/.kaggle", exist_ok=True)
os.rename("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 600)


In [ ]:
!pip install -q kaggle
!kaggle datasets download -d wordsforthewise/lending-club -p /content/data --unzip


In [ ]:
!ls /content/data


## 1. Carga y filtrado (Nicolás)

Se procesa el CSV en trozos (`chunksize`) para no saturar la memoria, filtrando por la ventana de tiempo del contrato (préstamos emitidos entre enero 2012 y junio 2014) y quedándonos solo con préstamos que ya tienen un desenlace final.

In [ ]:
import pandas as pd

RUTA = "/content/data/accepted_2007_to_2018Q4.csv.gz"
FINALES = ["Fully Paid", "Charged Off", "Default"]

partes = []
for chunk in pd.read_csv(RUTA, chunksize=200_000, low_memory=False, compression="gzip"):
    chunk["issue_d"] = pd.to_datetime(chunk["issue_d"], format="%b-%Y", errors="coerce")
    chunk = chunk[chunk["issue_d"].between("2012-01-01", "2014-06-30")]
    chunk = chunk[chunk["loan_status"].isin(FINALES)]
    partes.append(chunk)

df = pd.concat(partes, ignore_index=True)
df["target"] = (df["loan_status"] != "Fully Paid").astype(int)


### Features derivadas (compartidas por todo el equipo)

`term` y `emp_length` llegan como texto (`" 36 months"`, `"10+ years"`); se convierten a número. `fico_avg` promedia el rango de FICO. `hist_crediticio_anios` calcula cuántos años de historial crediticio tenía el solicitante al momento de pedir el préstamo.

In [ ]:
df["term"] = df["term"].str.extract(r"(\d+)")[0].astype(float)
df["emp_length"] = df["emp_length"].replace({"< 1 year": "0 years"}).str.extract(r"(\d+)")[0].astype(float)
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
cr = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["hist_crediticio_anios"] = (df["issue_d"] - cr).dt.days / 365.25


## 2. Calidad de datos (Nicolás)

In [ ]:
df.shape


In [ ]:
df.dtypes.value_counts()


In [ ]:
faltantes = df.isna().mean().sort_values(ascending=False) * 100
faltantes.head(30)


**Filas y columnas:** el dataset filtrado (préstamos emitidos entre enero 2012 y junio 2014, con desenlace final) quedó con 287,640 filas y 154 columnas. De esas, 118 son numéricas (117 float64 + 1 int64, equivalente al 76.6%), 35 son categóricas de tipo object (22.7%) y 1 es de tipo fecha (datetime64, 0.6%).

In [ ]:
print((faltantes == 100).sum(), "columnas con 100% de faltantes, de", len(faltantes), "columnas totales")


In [ ]:
import matplotlib.pyplot as plt

faltantes.head(20).plot(kind="barh", figsize=(8, 6))
plt.xlabel("% de valores faltantes")
plt.gca().invert_yaxis()
plt.show()


El gráfico muestra que 32 columnas tienen 100% de valores faltantes, principalmente las relacionadas al co-aplicante (`sec_app_*`) y la columna `member_id` (eliminada por Kaggle por motivos de privacidad). Esto se debe a que en este subconjunto de préstamos (2012–2014) casi no hubo solicitudes conjuntas, por lo que esas variables nunca se registraron.

Entre las columnas con 0% de faltantes se encuentran algunas como `total_pymnt` y `out_prncp_inv`, que corresponden al Grupo G3 (leakage) del contrato del equipo.

In [ ]:
df["target"].value_counts(normalize=True)


El target está desbalanceado: 83.6% de los préstamos se pagaron completos y 16.4% terminaron en impago o default.

Esto confirma por qué la accuracy sola no sirve como métrica principal en este problema: un modelo que siempre prediga "va a pagar" tendría una accuracy de 83.6% sin haber aprendido ningún patrón real. Se requieren métricas más informativas como ROC-AUC, PR-AUC y F1.

## 3. Exploración visual y outliers (Kris)

In [ ]:
import seaborn as sns

tasa_por_anio = df.groupby(df["issue_d"].dt.to_period("M"))["target"].mean()
tasa_por_anio.plot(figsize=(10, 4))
plt.ylabel("tasa de impago")
plt.title("Tasa de impago por mes de emisión")
plt.show()


In [ ]:
sns.barplot(data=df, x="grade", y="target", order=sorted(df["grade"].dropna().unique()))
plt.ylabel("tasa de impago")
plt.show()


In [ ]:
sns.barplot(data=df, y="purpose", x="target", estimator="mean",
            order=df.groupby("purpose")["target"].mean().sort_values(ascending=False).index)
plt.xlabel("tasa de impago")
plt.show()


In [ ]:
import numpy as np

df["log_annual_inc"] = df["annual_inc"].where(df["annual_inc"] > 0).apply(np.log10)
sns.histplot(data=df, x="log_annual_inc", hue="target", stat="density", common_norm=False, bins=40)
plt.show()


In [ ]:
bins = [300, 600, 660, 700, 740, 780, 850]
df["fico_bin"] = pd.cut(df["fico_avg"], bins=bins)
sns.barplot(data=df, x="fico_bin", y="target")
plt.xticks(rotation=45)
plt.ylabel("tasa de impago")
plt.show()


In [ ]:
numericas = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_avg", "revol_util", "target"]
sns.heatmap(df[numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.show()


### Tasa de impago en el tiempo
La tasa de impago presenta variaciones y tendencias a lo largo de los meses de emisión. Esto evidencia que las condiciones cambian con el tiempo y justifica directamente por qué la partición debe ser temporal (entrenar con el pasado, predecir el futuro) y no aleatoria.

### Grade y Purpose
Existe una relación directa y progresiva entre el grado del préstamo y el riesgo. Los préstamos con calificación 'A' tienen la menor tasa de impago, la cual aumenta escalonadamente hasta llegar a la calificación 'G' (el mayor riesgo). El propósito del préstamo también influye de manera considerable: ciertas categorías son inherentemente más riesgosas.

### Ingreso y FICO
En escala logarítmica, los prestatarios que caen en impago tienden a tener ingresos ligeramente inferiores a los que pagan su deuda. Hay una fuerte tendencia a la baja en la tasa de impago a medida que aumenta el puntaje FICO; las bandas más altas presentan los niveles de riesgo más bajos de todo el dataset.

### Registros sospechosos

In [ ]:
sospechosos = {
    "annual_inc <= 0 o > 1,000,000": ((df["annual_inc"] <= 0) | (df["annual_inc"] > 1_000_000)).sum(),
    "dti < 0 o > 60": ((df["dti"] < 0) | (df["dti"] > 60)).sum(),
    "revol_util > 100": (df["revol_util"] > 100).sum(),
    "hist_crediticio negativo": (df["hist_crediticio_anios"] < 0).sum(),
}
sospechosos


Encontramos registros físicamente imposibles: ingresos iguales o menores a cero, ratios de endeudamiento (DTI) negativos o mayores a 60, e historiales crediticios negativos. Como `revol_util > 100` puede ser válido en casos raros de sobregiro, no se elimina de inmediato, pero estos outliers requerirán limpieza antes de entrenar el modelo final.

## 4. Leakage, variables y sesgos (Rubí)

### Evidencia numérica del leakage

Entrenamos la misma regresión logística dos veces sobre el mismo train/validación, cambiando solo las variables de entrada.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

np.random.seed(42)
train = df[df["issue_d"] < "2014-01-01"]
valid = df[df["issue_d"] >= "2014-01-01"]

G1_NUM = ["loan_amnt", "term", "emp_length", "annual_inc", "dti", "delinq_2yrs", "fico_avg",
          "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc",
          "hist_crediticio_anios"]
G1_CAT = ["home_ownership", "verification_status", "purpose", "addr_state"]
G3_NUM = ["total_pymnt", "total_rec_prncp", "recoveries", "last_pymnt_amnt"]

def pipeline_leakage(num, cat):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

def auc_de(num, cat):
    modelo = pipeline_leakage(num, cat).fit(train[num + cat], train["target"])
    p = modelo.predict_proba(valid[num + cat])[:, 1]
    return roc_auc_score(valid["target"], p)

auc_limpio = auc_de(G1_NUM, G1_CAT)
auc_con_leakage = auc_de(G1_NUM + G3_NUM, G1_CAT)
auc_limpio, auc_con_leakage


**Comparación:** al agregar variables que solo existen después de que el préstamo terminó (`total_pymnt`, `total_rec_prncp`, `recoveries`, `last_pymnt_amnt`), el AUC sube de **0.6858** a **0.9996**. Ese salto artificial demuestra que esas columnas filtran directamente la respuesta en vez de aportar señal legítima de riesgo.

### Tabla de clasificación de variables (G0-G3)

In [ ]:
grupos = {
    "G0": [
        "id", "member_id", "url", "desc", "title", "emp_title", "zip_code", "policy_code"
    ],
    "G1": G1_NUM + G1_CAT + [
        "funded_amnt", "funded_amnt_inv", "initial_list_status", "application_type",
        "disbursement_method",
        "mths_since_last_delinq", "mths_since_last_record", "mths_since_last_major_derog",
        "collections_12_mths_ex_med", "chargeoff_within_12_mths", "tax_liens",
        "pub_rec_bankruptcies", "acc_now_delinq", "delinq_amnt",
        "tot_coll_amt", "tot_cur_bal", "tot_hi_cred_lim", "total_bal_ex_mort",
        "total_bc_limit", "total_il_high_credit_limit", "total_rev_hi_lim",
        "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy", "bc_util",
        "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op",
        "mo_sin_rcnt_tl", "mort_acc",
        "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
        "mths_since_recent_revol_delinq",
        "num_accts_ever_120_pd", "num_actv_bc_tl", "num_actv_rev_tl", "num_bc_sats",
        "num_bc_tl", "num_il_tl", "num_op_rev_tl", "num_rev_accts",
        "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
        "num_tl_90g_dpd_24m", "num_tl_op_past_12m",
        "pct_tl_nvr_dlq", "percent_bc_gt_75",
        "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "open_rv_12m", "open_rv_24m",
        "mths_since_rcnt_il", "total_bal_il", "il_util", "max_bal_bc", "all_util",
        "inq_fi", "total_cu_tl", "inq_last_12m",
        "annual_inc_joint", "dti_joint", "verification_status_joint",
        "sec_app_fico_range_low", "sec_app_fico_range_high", "sec_app_earliest_cr_line",
        "sec_app_inq_last_6mths", "sec_app_mort_acc", "sec_app_open_acc",
        "sec_app_revol_util", "sec_app_open_act_il", "sec_app_num_rev_accts",
        "sec_app_chargeoff_within_12_mths", "sec_app_collections_12_mths_ex_med",
        "sec_app_mths_since_last_major_derog",
    ],
    "G2": ["grade", "sub_grade", "int_rate", "installment"],
    "G3": [
        "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
        "total_rec_late_fee", "recoveries", "collection_recovery_fee",
        "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d", "last_credit_pull_d",
        "out_prncp", "out_prncp_inv",
        "last_fico_range_high", "last_fico_range_low",
        "hardship_flag", "hardship_type", "hardship_reason", "hardship_status",
        "deferral_term", "hardship_amount", "hardship_start_date", "hardship_end_date",
        "payment_plan_start_date", "hardship_length", "hardship_dpd",
        "hardship_loan_status", "orig_projected_additional_accrued_interest",
        "hardship_payoff_balance_amount", "hardship_last_payment_amount",
        "debt_settlement_flag", "debt_settlement_flag_date", "settlement_status",
        "settlement_date", "settlement_amount", "settlement_percentage", "settlement_term",
    ],
}

tabla_variables = []
for grupo, cols in grupos.items():
    for c in cols:
        tabla_variables.append({"columna": c, "grupo": grupo})
tabla_variables = pd.DataFrame(tabla_variables)

tabla_variables = pd.concat([tabla_variables, pd.DataFrame([
    {"columna": "earliest_cr_line", "grupo": "G1"},
    {"columna": "fico_range_low", "grupo": "G1"},
    {"columna": "fico_range_high", "grupo": "G1"},
    {"columna": "pymnt_plan", "grupo": "G1"},
    {"columna": "revol_bal_joint", "grupo": "G1"},
])], ignore_index=True)

clasificadas = set(tabla_variables["columna"])
sin_clasificar = [c for c in df.columns if c not in clasificadas]
print(f"Columnas clasificadas: {len(clasificadas)}")
print(f"Columnas sin clasificar ({len(sin_clasificar)}):")
print(sin_clasificar)

tabla_variables.to_csv("/content/data/tabla_variables.csv", index=False)
tabla_variables.head(20)


## Sesgos, limitaciones y riesgos éticos del proyecto

**1. Sesgo de selección.** Solo trabajamos con préstamos que Lending Club ya aceptó. Los rechazados existen en `rejected_2007_to_2018Q4.csv`, pero no los usamos, así que nuestro modelo no sabe nada de esa población y no podemos evaluar cómo se comportaría ahí. En la práctica, esto significa que no podemos decir que el modelo "decide a quién prestar" en general — solo ordena el riesgo dentro de la gente que Lending Club ya consideró aceptable. Para la entrega final nos gustaría al menos mirar el dataset de rechazados, aunque sea sin etiqueta.

**2. Variables que pueden actuar como proxy.** No tenemos raza, género ni edad en el dataset, pero `addr_state` y `annual_inc` pueden correlacionar con esas características por temas históricos (segregación residencial, brecha salarial). Si no tenemos cuidado, el modelo podría terminar discriminando indirectamente. Antes de pensar en llevar esto a producción, habría que revisar la tasa de aprobación por estado y por rango de ingreso, y probar qué tanto AUC se pierde si sacamos `addr_state`.

**3. Datos que el propio solicitante declara.** `annual_inc` y `emp_length` los reporta la persona que pide el préstamo, y no siempre están verificados — de hecho por eso existe la columna `verification_status`. Una idea para la siguiente etapa es cruzar `annual_inc` con `verification_status`, para que el modelo aprenda a confiar menos en ingresos no verificados.

**4. Ventana de tiempo corta.** Entrenamos solo con préstamos de EE.UU. entre 2012 y 2014. La economía cambia, así que no hay garantía de que esto funcione igual en otra época (una recesión, por ejemplo) o en otro país.

**5. Dependemos en parte del criterio de Lending Club.** `grade` e `int_rate` (nuestro grupo G2) ya son un puntaje de riesgo que la plataforma calculó con información que nosotros no tenemos. Por eso reportamos el AUC con y sin esas variables: si el salto es grande, hay que ser honestos en que parte de lo que "aprende" el modelo es imitar ese puntaje, no calcular riesgo desde cero.

**6. Las clases están desbalanceadas.** 83.6% de los préstamos se pagó y 16.4% no. Un modelo que siempre diga "va a pagar" ya acierta el 83.6% de las veces sin haber entendido nada, así que la accuracy sola no nos sirve de mucho — por eso priorizamos ROC-AUC y PR-AUC, y usamos `class_weight="balanced"` en la regresión logística.

**7. No tratamos los outliers todavía.** Encontramos `dti` negativo, `revol_util` por encima de 100% y algunos ingresos anuales en millones. Puede que parte sea error de captura y parte casos reales extremos; por ahora los dejamos así, pero en la siguiente etapa vamos a acotarlos y medir cuánto cambia el AUC.

## 5. Partición temporal y baseline oficial (Valentino)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix, accuracy_score

NUM = G1_NUM
CAT = G1_CAT
LC_NUM, LC_CAT = ["int_rate", "installment"], ["grade"]

def pipeline_baseline(num, cat):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])
    return Pipeline([("pre", pre),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])

def evaluar(nombre, num, cat):
    modelo = pipeline_baseline(num, cat).fit(train[num + cat], train["target"])
    p = modelo.predict_proba(valid[num + cat])[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        "modelo": nombre,
        "roc_auc": roc_auc_score(valid["target"], p),
        "pr_auc": average_precision_score(valid["target"], p),
        "f1": f1_score(valid["target"], pred),
        "accuracy": accuracy_score(valid["target"], pred),
        "matriz_confusion": confusion_matrix(valid["target"], pred).tolist(),
    }

dummy = DummyClassifier(strategy="most_frequent", random_state=42).fit(train[NUM], train["target"])
pred_dummy = dummy.predict(valid[NUM])

import json
resultados = [
    {"modelo": "dummy", "accuracy": accuracy_score(valid["target"], pred_dummy), "roc_auc": 0.5},
    evaluar("logreg_G1", NUM, CAT),
    evaluar("logreg_G1_mas_G2", NUM + LC_NUM, CAT + LC_CAT),
]

with open("/content/data/metrics_baseline.json", "w") as f:
    json.dump(resultados, f, indent=2)

print(json.dumps(resultados, indent=2))


**Lectura del resultado:** el baseline con variables G1 alcanza un ROC-AUC de 0.685, superando claramente al modelo aleatorio (0.500). Esto significa que, tomando al azar un préstamo que no se pagó y uno que sí, el modelo asigna mayor riesgo al que no se pagó el 68.5% de las veces. Al agregar las variables propietarias `grade` e `int_rate` (G2), el AUC sube a 0.699; este ligero incremento tiene sentido porque dichas variables representan un puntaje de riesgo que Lending Club ya calculó internamente, facilitando la tarea predictiva al proveer información preprocesada por la plataforma.

## 6. Guardamos dataset procesado

In [ ]:
df.drop(columns=["fico_bin"], errors="ignore").to_parquet("/content/data/lc_2012_2014H1.parquet", index=False)